In [ ]:
# --- Imports and utility function ---
import os
import numpy as np
import pandas as pd
import requests
from io import StringIO

def fetch_knmi_prec_evap(station: int, start_date: str, end_date: str):
    """
    Fetch KNMI daily data and compute precipitation and Makkink evapotranspiration.

    Parameters:
    - station (int): KNMI station number (e.g., 249 for Berkhout)
    - start_date (str): Start date in 'YYYY-MM-DD' or 'YYYYMMDD' format
    - end_date (str): End date in 'YYYY-MM-DD' or 'YYYYMMDD' format

    Returns:
    - prec (pd.Series): Precipitation series in mm/day (float64)
    - evap (pd.Series): Makkink evapotranspiration series in mm/day (float64)
    """
    start = start_date.replace('-', '')
    end = end_date.replace('-', '')
    url = 'https://www.daggegevens.knmi.nl/klimatologie/daggegevens'
    params = {
        'start': start,
        'end': end,
        'stns': str(station),
        'vars': 'Q:RH:TG',
        'fmt': 'csv'
    }
    response = requests.post(url, data=params)
    response.raise_for_status()
    csv_data = '\n'.join(line for line in response.text.splitlines() if not line.startswith('#'))

    knmi_df = pd.read_csv(StringIO(csv_data), header=None)
    knmi_df.columns = ['STN', 'DATE', 'Q', 'RH', 'TG']
    knmi_df['DATE'] = pd.to_datetime(knmi_df['DATE'], format='%Y%m%d')

    # Unit conversions
    Rs_MJ = knmi_df['Q'].clip(lower=0) * 0.01      # J/cm² → MJ/m²
    T_C = knmi_df['TG'] / 10.0                      # 0.1 °C → °C
    knmi_df['RH'] = knmi_df['RH'].where(knmi_df['RH'] >= 0, 0) / 10.0  # 0.1 mm → mm (-1 = trace → 0)

    # Makkink evapotranspiration
    gamma = 0.066       # psychrometric constant (kPa/°C)
    lambda_MJ = 2.45    # latent heat of vaporization (MJ/m² per mm)

    delta = (
        4098
        * (0.6108 * np.exp((17.27 * T_C) / (T_C + 237.3)))
        / ((T_C + 237.3) ** 2)
    )

    knmi_df['ET'] = (
        0.65
        * (delta / (delta + gamma))
        * (Rs_MJ / lambda_MJ)
    ).clip(lower=0)

    knmi_df = knmi_df.set_index('DATE')
    prec = knmi_df['RH'].astype(float)
    evap = knmi_df['ET'].astype(float)
    return prec, evap

In [2]:
# --- Discover repo root and set up paths ---
from pathlib import Path
import os

# Find repo root by looking for pyproject.toml or .git
repo_root = Path.cwd()
for candidate in [repo_root] + list(repo_root.parents):
    if (candidate / 'pyproject.toml').exists() or (candidate / '.git').exists():
        repo_root = candidate
        break
print('Repo root:', repo_root)

# Set output directory relative to repo root
output_dir = repo_root / 'input_stressors'
os.makedirs(output_dir, exist_ok=True)
print('Output directory:', output_dir)


Repo root: d:\Users\jvanruitenbeek\data_validation
Output directory: d:\Users\jvanruitenbeek\data_validation\input_stressors


In [3]:
# --- Parameters ---
# Define the station and date range for the data pull.
station = 249  # Berkhout
start_date = '2020-01-01'
end_date = '2026-06-23'

In [4]:
# --- Download and save data ---
from datetime import datetime

prec, evap = fetch_knmi_prec_evap(station, start_date, end_date)

# Get current timestamp for filenames
timestamp = datetime.now().strftime('%Y%m%d%H%M%S')

# Save precipitation
prec_filename = f'prec_station_{station}_{timestamp}.csv'
prec_path = output_dir / prec_filename
prec.to_csv(prec_path, header=True)
print(f'Precipitation saved to: {prec_path}')

# Save evapotranspiration
evap_filename = f'evap_station_{station}_{timestamp}.csv'
evap_path = output_dir / evap_filename
evap.to_csv(evap_path, header=True)
print(f'Evapotranspiration saved to: {evap_path}')


Precipitation saved to: d:\Users\jvanruitenbeek\data_validation\input_stressors\prec_station_249_20260624132740.csv
Evapotranspiration saved to: d:\Users\jvanruitenbeek\data_validation\input_stressors\evap_station_249_20260624132740.csv


In [ ]:
# --- Yearly cumulative plot and summary ---
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

df = pd.DataFrame({'Precipitation': prec, 'Makkink ET': evap})
df['year'] = df.index.year
df['day_of_year'] = df.index.dayofyear

# Print yearly sums
yearly_sums = df.groupby('year')[['Precipitation', 'Makkink ET']].sum()
for year, row in yearly_sums.iterrows():
    print(f"{year}:  Precipitation = {row['Precipitation']:.1f} mm,  Makkink ET = {row['Makkink ET']:.1f} mm")

# Compute cumulative sums per year
df['cum_prec'] = df.groupby('year')['Precipitation'].cumsum()
df['cum_evap'] = df.groupby('year')['Makkink ET'].cumsum()

years = sorted(df['year'].unique())
colors = ['#2563EB', '#D97706', '#059669', '#DC2626', '#7C3AED', '#0891B2', '#BE185D']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

for i, year in enumerate(years):
    subset = df[df['year'] == year]
    c = colors[i % len(colors)]
    ax1.plot(subset['day_of_year'], subset['cum_prec'], color=c, linewidth=1.5, label=str(year))
    ax2.plot(subset['day_of_year'], subset['cum_evap'], color=c, linewidth=1.5, label=str(year))

for ax, title in [(ax1, 'Cumulative Precipitation'), (ax2, 'Cumulative Makkink Evapotranspiration')]:
    ax.set_ylabel('mm')
    ax.set_title(title)
    ax.legend(loc='upper left', frameon=False, ncol=len(years))
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

ax2.set_xlabel('Day of year')
ax2.xaxis.set_major_locator(mticker.MultipleLocator(30))
plt.tight_layout()
plt.show()

### Ophalen uurlijkse datasets